In [1]:
# ============================================================
#  YouTube Lecture Assistant — Module 1: Data Extractor
#  Course  : Programming for AI
#  Purpose : Extract a clean transcript from any YouTube URL.
#            Tries the fast API route first; falls back to
#            Whisper-based audio transcription if needed.
# ============================================================

# ── 1. INSTALL DEPENDENCIES (Google Colab) ──────────────────
# Run this cell first in Colab, then restart the runtime.

!pip install -q youtube-transcript-api
!pip install -q yt-dlp
!pip install -q transformers
!pip install -q torch
!pip install -q ffmpeg-python

# For Colab, ffmpeg binary is usually pre-installed.
# If not, uncomment:
# !apt-get install -y -q ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 22.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.5 MB/s eta 0:00:00


In [2]:
# ── 2. IMPORTS ───────────────────────────────────────────────
import os
import re
import textwrap

# YouTube transcript (fast path)
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
)

# Audio download (fallback path)
import yt_dlp

# Whisper ASR via Hugging Face Transformers (fallback path)
from transformers import pipeline

In [3]:
# ── 3. HELPERS ───────────────────────────────────────────────

def _extract_video_id(youtube_url: str) -> str:
    """
    Parse the 11-character video ID from a variety of YouTube URL formats.

    Supports:
      - https://www.youtube.com/watch?v=VIDEO_ID
      - https://youtu.be/VIDEO_ID
      - https://www.youtube.com/embed/VIDEO_ID
      - https://youtube.com/shorts/VIDEO_ID

    Args:
        youtube_url: Any standard YouTube link.

    Returns:
        The 11-character video ID string.

    Raises:
        ValueError: If the ID cannot be found in the URL.
    """
    # Regex covers all common YouTube URL patterns
    pattern = (
        r"(?:v=|youtu\.be/|embed/|shorts/|/v/|/e/|watch\?v=|&v=)"
        r"([A-Za-z0-9_-]{11})"
    )
    match = re.search(pattern, youtube_url)
    if not match:
        raise ValueError(
            f"Could not extract a valid YouTube video ID from URL:\n  {youtube_url}\n"
            "Please make sure you are using a standard YouTube link."
        )
    return match.group(1)

In [4]:
def _clean_text(raw_text: str) -> str:
    """
    Normalise raw transcript text into a clean, readable paragraph string.

    Steps performed:
      1. Strip leading/trailing whitespace from every line.
      2. Remove leftover SRT/VTT timestamp lines  e.g. "00:01:23,456 --> 00:01:26,789"
      3. Remove bare sequence numbers (SRT artefacts).
      4. Collapse multiple blank lines into a single blank line.
      5. Join lines so the result is one continuous block of text.
      6. Collapse runs of whitespace into a single space.

    Args:
        raw_text: Text that may contain timestamps, newlines, or other noise.

    Returns:
        A single, clean string of plain text.
    """
    lines = raw_text.splitlines()

    cleaned_lines = []
    for line in lines:
        line = line.strip()

        # Skip SRT/VTT timestamp arrows  "00:00:01,000 --> 00:00:04,000"
        if re.match(r"^\d{1,2}:\d{2}:\d{2}[,\.]\d{3}\s*-->\s*\d{1,2}:\d{2}:\d{2}", line):
            continue

        # Skip bare numeric sequence numbers used in SRT files
        if re.match(r"^\d+$", line):
            continue

        # Skip WebVTT header
        if line.upper().startswith("WEBVTT"):
            continue

        cleaned_lines.append(line)

    # Join everything, then collapse excess whitespace
    joined = " ".join(filter(None, cleaned_lines))          # drop empty strings
    clean = re.sub(r"\s{2,}", " ", joined).strip()          # collapse whitespace
    return clean

In [5]:
def _fetch_transcript_via_api(video_id: str) -> str | None:
    """
    Attempt to fetch a transcript using youtube-transcript-api (fast path).

    Preference order:
      1. Manually uploaded English transcript  (most accurate)
      2. Auto-generated English transcript     (usually available)
      3. Any available transcript language     (last resort — may need translation)

    Args:
        video_id: The 11-character YouTube video identifier.

    Returns:
        The concatenated raw transcript text, or None if unavailable.
    """
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)

        # ── Try manually created English first ──
        try:
            transcript = transcript_list.find_manually_created_transcript(["en"])
            print("  [Transcript API] Found manually created English transcript.")
        except NoTranscriptFound:
            pass
        else:
            segments = transcript.fetch()
            return " ".join(seg["text"] for seg in segments)

        # ── Fall back to auto-generated English ──
        try:
            transcript = transcript_list.find_generated_transcript(["en"])
            print("  [Transcript API] Found auto-generated English transcript.")
        except NoTranscriptFound:
            pass
        else:
            segments = transcript.fetch()
            return " ".join(seg["text"] for seg in segments)

        # ── Accept any language as a last resort ──
        for transcript in transcript_list:
            print(
                f"  [Transcript API] Using transcript in language: "
                f"'{transcript.language}' (code: {transcript.language_code})."
            )
            segments = transcript.fetch()
            return " ".join(seg["text"] for seg in segments)

    except TranscriptsDisabled:
        print("  [Transcript API] Transcripts are disabled for this video.")
    except Exception as exc:                              # network errors, etc.
        print(f"  [Transcript API] Unexpected error: {exc}")

    return None   # Signal to caller that this path failed

In [6]:
def _download_audio(youtube_url: str, output_path: str = "lecture_audio") -> str:
    """
    Download the best available audio track from YouTube as a WAV file.

    Uses yt-dlp with the ffmpeg post-processor to convert the stream to
    16 kHz mono WAV, which is the format Whisper expects.

    Args:
        youtube_url  : The full YouTube URL.
        output_path  : Filename prefix for the downloaded audio (no extension).

    Returns:
        The full path to the saved .wav file.

    Raises:
        RuntimeError: If the download or conversion fails.
    """
    print("  [yt-dlp] Downloading audio track …")

    ydl_opts = {
        # Extract only audio; no video download needed
        "format": "bestaudio/best",

        # Convert to WAV at 16 kHz mono (ideal for Whisper)
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "wav",
                "preferredquality": "192",
            }
        ],

        # Output template — yt-dlp appends ".wav" after conversion
        "outtmpl": output_path,

        # Suppress verbose yt-dlp console output
        "quiet": True,
        "no_warnings": True,

        # Extra ffmpeg args to force 16 kHz mono
        "postprocessor_args": ["-ar", "16000", "-ac", "1"],
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url])
    except Exception as exc:
        raise RuntimeError(f"yt-dlp download failed: {exc}") from exc

    wav_file = output_path + ".wav"
    if not os.path.isfile(wav_file):
        raise RuntimeError(
            f"Audio download succeeded but expected file not found: {wav_file}"
        )

    print(f"  [yt-dlp] Audio saved → {wav_file}")
    return wav_file

In [7]:
def _transcribe_with_whisper(audio_path: str, model_id: str = "openai/whisper-small") -> str:
    """
    Transcribe a local audio file using a Hugging Face Whisper model.

    The pipeline runs entirely locally — no API key required.

    Args:
        audio_path : Path to the WAV (or MP3) file to transcribe.
        model_id   : Hugging Face model identifier.
                     "openai/whisper-small"  — good accuracy, ~244 MB
                     "openai/whisper-base"   — faster, ~74 MB, slightly less accurate
                     "openai/whisper-medium" — best quality, ~769 MB (slow on CPU)

    Returns:
        The transcribed text as a single string.

    Raises:
        RuntimeError: If the Whisper pipeline fails.
    """
    print(f"  [Whisper] Loading model '{model_id}' … (first run downloads weights)")

    try:
        # chunk_length_s=30 lets the pipeline handle audio longer than 30 s
        asr_pipeline = pipeline(
            task="automatic-speech-recognition",
            model=model_id,
            chunk_length_s=30,          # process audio in 30-second windows
            stride_length_s=5,          # overlap to avoid boundary cut-offs
        )

        print("  [Whisper] Transcribing audio … this may take several minutes on CPU.")
        result = asr_pipeline(audio_path, return_timestamps=False)

    except Exception as exc:
        raise RuntimeError(f"Whisper transcription failed: {exc}") from exc

    # The pipeline returns {"text": "..."} for single files
    return result.get("text", "")


In [8]:
# ── 4. MAIN PUBLIC FUNCTION ───────────────────────────────────

def extract_lecture_text(
    youtube_url: str,
    save_to_file: bool = True,
    output_filename: str = "transcript.txt",
    whisper_model: str = "openai/whisper-small",
) -> str:
    """
    Extract the full lecture text from a YouTube video.

    Strategy
    --------
    1. **Fast path** — Attempt to pull the transcript directly via
       ``youtube-transcript-api``.  This works for most lectures where
       the uploader has captions or YouTube generated auto-captions.
       No audio download or heavy model inference needed.

    2. **Fallback path** — If transcripts are disabled or missing:
       a. Download the audio as a WAV file using ``yt-dlp``.
       b. Transcribe locally with OpenAI's Whisper model via
          Hugging Face ``transformers``.

    In both cases the text is cleaned before being returned.

    Args:
        youtube_url    : Any valid YouTube video URL.
        save_to_file   : If True, writes the transcript to ``output_filename``.
        output_filename: Target filename for the saved transcript.
        whisper_model  : Hugging Face model ID for the fallback Whisper path.

    Returns:
        A single clean string containing the full lecture transcript.

    Raises:
        ValueError  : If the URL does not contain a valid video ID.
        RuntimeError: If both extraction paths fail.

    Example:
        >>> text = extract_lecture_text("https://www.youtube.com/watch?v=dQw4w9WgXcQ")
        >>> print(text[:300])
    """

    print("=" * 60)
    print("  YouTube Lecture Assistant — Module 1: Data Extractor")
    print("=" * 60)
    print(f"  URL : {youtube_url}\n")
 # ── Step 1: Validate URL and extract video ID ──────────────
    video_id = _extract_video_id(youtube_url)
    print(f"  Video ID detected : {video_id}\n")

    raw_text: str | None = None   # will hold the unclean transcript
# ── Step 2: Try the fast API path first ───────────────────
    print("[ PATH 1 ] Fetching transcript via youtube-transcript-api …")
    raw_text = _fetch_transcript_via_api(video_id)
   # ── Step 3: Fallback — download audio + Whisper ASR ───────
    if raw_text is None:
        print(
            "\n[ PATH 2 ] Transcript unavailable. "
            "Falling back to audio download + Whisper ASR …"
        )

        audio_file = None
        try:
            audio_file = _download_audio(youtube_url, output_path="lecture_audio")
            raw_text = _transcribe_with_whisper(audio_file, model_id=whisper_model)

        except RuntimeError as err:
            # Clean up partial audio file if it exists
            if audio_file and os.path.isfile(audio_file):
                os.remove(audio_file)
            raise RuntimeError(
                "Both extraction paths failed.\n"
                f"  Whisper error: {err}\n"
                "Please check your internet connection and that ffmpeg is installed."
            ) from err

        finally:
            # Always delete the (potentially large) audio file after transcription
            if audio_file and os.path.isfile(audio_file):
                os.remove(audio_file)
                print(f"  [Cleanup] Removed temporary audio file: {audio_file}")

# ── Step 4: Clean the extracted text ──────────────────────
    print("\n[ STEP 3 ] Cleaning transcript text …")
    clean_text = _clean_text(raw_text)

    word_count = len(clean_text.split())
    char_count = len(clean_text)
    print(f"  Done. {word_count:,} words | {char_count:,} characters extracted.")

   # ── Step 5: Optionally save to file ───────────────────────
    if save_to_file:
        with open(output_filename, "w", encoding="utf-8") as f:
            # Wrap long lines at 100 chars for human readability
            wrapped = textwrap.fill(clean_text, width=100)
            f.write(wrapped)
        print(f"\n  Transcript saved → {output_filename}")

    print("=" * 60)
    print("  Extraction complete.")
    print("=" * 60)

    return clean_text


# ── 5. ENTRY POINT (for direct script execution) ─────────────

if __name__ == "__main__":
    # ----------------------------------------------------------------
    # Replace this URL with any YouTube lecture link to test the module.
    # ----------------------------------------------------------------
    TEST_URL = "https://www.youtube.com/watch?v=5sLYAQS9sWQ&t=44s"   # LLM

    try:
        lecture_text = extract_lecture_text(
            youtube_url=TEST_URL,
            save_to_file=True,
            output_filename="transcript.txt",
            whisper_model="openai/whisper-small",   # change to "whisper-base" for speed
        )

        # Preview the first 500 characters
        print("\n── PREVIEW (first 500 chars) ──")
        print(lecture_text[:500])
        print("…")

    except (ValueError, RuntimeError) as e:
        print(f"\n[ERROR] {e}")


  YouTube Lecture Assistant — Module 1: Data Extractor
  URL : https://www.youtube.com/watch?v=5sLYAQS9sWQ&t=44s

  Video ID detected : 5sLYAQS9sWQ

[ PATH 1 ] Fetching transcript via youtube-transcript-api …
  [Transcript API] Unexpected error: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'

[ PATH 2 ] Transcript unavailable. Falling back to audio download + Whisper ASR …
  [yt-dlp] Downloading audio track …
  [yt-dlp] Audio saved → lecture_audio.wav
  [Whisper] Loading model 'openai/whisper-small' … (first run downloads weights)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


  [Whisper] Transcribing audio … this may take several minutes on CPU.


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits pr

  [Cleanup] Removed temporary audio file: lecture_audio.wav

[ STEP 3 ] Cleaning transcript text …
  Done. 730 words | 4,239 characters extracted.

  Transcript saved → transcript.txt
  Extraction complete.

── PREVIEW (first 500 chars) ──
GPT or Generative Pretrained Transformer is a large language model or an LLM that can generate human-like text. And I've been using GPT in its various forms for years. In this video, we are going to, number one, ask what is an LLM. Number two, we are going to describe how they work. And then number three, we're going to ask what are the business applications of LLMs? So let's start with number one, what is a large language model? Well, a large language model is an instance of something else call
…


# Module 2

In [9]:
# ── Install dependencies (run once in Colab) ────────────────
!pip install transformers sentence-transformers faiss-cpu pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.4 MB/s eta 0:00:00


In [10]:
import re
import pandas as pd
import numpy as np
import faiss
from transformers import BartTokenizer, BartForConditionalGeneration
from sentence_transformers import SentenceTransformer

In [11]:
# ════════════════════════════════════════════════════════════
# INTERNAL HELPERS
# ════════════════════════════════════════════════════════════

def _split_into_sentences(text: str) -> list[str]:
    """
    Split text into individual sentences using punctuation boundaries.
    Handles common abbreviations gracefully.
    """
    # Split on sentence-ending punctuation followed by whitespace + capital letter
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    # Drop empty strings
    return [s.strip() for s in sentences if s.strip()]

In [12]:
def _chunk_text_for_summarizer(text: str, max_words: int = 400) -> list[str]:
    """
    Split a long text into word-count-bounded chunks for the summarization model.
    BART-large-CNN accepts roughly 1024 tokens (~700-800 words), but 400 words
    gives a comfortable safety margin while keeping chunks meaningful.
    """
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(current_chunk) >= max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = []

    if current_chunk:                          # flush the last partial chunk
        chunks.append(" ".join(current_chunk))

    return chunks



In [13]:
def _chunk_text_for_qa(
    text: str,
    sentences_per_chunk: int = 4,
    overlap: int = 1
) -> list[str]:
    """
    Split the transcript into overlapping sentence-window chunks for Q&A retrieval.

    Args:
        text:                 Raw transcript string.
        sentences_per_chunk:  Window size in sentences (default 4).
        overlap:              Number of sentences shared between consecutive chunks (default 1).

    Returns:
        List of chunk strings.
    """
    sentences = _split_into_sentences(text)
    if not sentences:
        return []

    step = sentences_per_chunk - overlap       # how far to advance each iteration
    chunks = []

    for i in range(0, len(sentences), step):
        window = sentences[i : i + sentences_per_chunk]
        chunks.append(" ".join(window))
        if i + sentences_per_chunk >= len(sentences):
            break                              # last window reached

    return chunks


In [14]:
# ════════════════════════════════════════════════════════════
# PUBLIC FUNCTION 1 — SUMMARIZATION  (Task A)
# ════════════════════════════════════════════════════════════

def generate_lecture_summary(text: str, output_file: str = "summary.txt") -> str:
    """
    Summarize a long lecture transcript using facebook/bart-large-cnn.

    Strategy for long texts:
        1. Split the transcript into <=400-word chunks.
        2. Summarize each chunk individually.
        3. Concatenate the partial summaries.
        4. If the combined partial summaries are still long,
           run a final compression pass to produce one coherent summary.

    Args:
        text:        Full lecture transcript string (from Module 1).
        output_file: Path to save the final summary (default: summary.txt).

    Returns:
        The final summary string.
    """
    print("=" * 60)
    print("TASK A - SUMMARIZATION")
    print("=" * 60)

    if not text or not text.strip():
        raise ValueError("Input text is empty. Please pass a valid transcript.")

    # ── Load model & tokenizer explicitly ────────────────────
    MODEL_NAME = "facebook/bart-large-cnn"
    print(f"[1/4] Loading tokenizer and model ({MODEL_NAME})...")
    tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
    model     = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.eval()
    print("      Model loaded successfully.")

    def _summarize_chunk(chunk_text: str, max_len: int = 150, min_len: int = 40) -> str:
        """Tokenize one text chunk and return its summary string."""
        inputs = tokenizer(
            chunk_text,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=max_len,
            min_length=min_len,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )
        return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # ── Chunk the transcript ─────────────────────────────────
    print("[2/4] Splitting transcript into chunks...")
    chunks = _chunk_text_for_summarizer(text, max_words=400)
    print(f"      {len(chunks)} chunk(s) created.")

    # ── Summarize each chunk ─────────────────────────────────
    print("[3/4] Summarizing each chunk...")
    partial_summaries = []

    for idx, chunk in enumerate(chunks, start=1):
        print(f"      Chunk {idx}/{len(chunks)}...", end=" ", flush=True)
        partial_summaries.append(_summarize_chunk(chunk))
        print("done.")

    combined = " ".join(partial_summaries)

    # ── Optional second-pass summary ─────────────────────────
    final_summary = combined
    if len(combined.split()) > 400:
        print("      Combined summaries are long - running final compression pass...")
        final_summary = _summarize_chunk(combined, max_len=300, min_len=80)
        print("      Final pass complete.")

    # ── Save to file ─────────────────────────────────────────
    print(f"[4/4] Saving summary to '{output_file}'...")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(final_summary)
    print(f"      Saved. Summary length: {len(final_summary.split())} words.")

    print("\n-- SUMMARY PREVIEW (first 300 chars) --")
    print(final_summary[:300] + ("..." if len(final_summary) > 300 else ""))
    print("=" * 60 + "\n")

    return final_summary

In [15]:
# ════════════════════════════════════════════════════════════
# PUBLIC FUNCTION 2 — CHUNKING + VECTORIZATION + FAISS  (Tasks B & C)
# ════════════════════════════════════════════════════════════

def create_faiss_index(
    text: str,
    chunks_file: str = "lecture_chunks.csv",
    index_file: str = "lecture_index.index",
    sentences_per_chunk: int = 4,
    overlap: int = 1
) -> tuple[pd.DataFrame, faiss.Index]:
    """
    Split the transcript into overlapping chunks, embed them with
    sentence-transformers/all-MiniLM-L6-v2, and store them in a FAISS index.

    Task B — Chunking:
        Splits the transcript into overlapping sentence windows and saves
        the result to a Pandas DataFrame → lecture_chunks.csv.

    Task C — Vectorization & FAISS:
        Encodes every chunk into a 384-dimensional embedding, builds a
        faiss.IndexFlatL2 (exact L2-distance search), and saves the index
        to lecture_index.index.

    Args:
        text:               Full transcript string (from Module 1).
        chunks_file:        CSV output path for the chunk DataFrame.
        index_file:         Path for the saved FAISS index file.
        sentences_per_chunk: Window size (default 4 sentences per chunk).
        overlap:            Sentence overlap between consecutive chunks (default 1).

    Returns:
        (df, index) — the Pandas DataFrame and the populated FAISS index.
    """
    print("=" * 60)
    print("TASK B — CHUNKING FOR Q&A")
    print("=" * 60)

    if not text or not text.strip():
        raise ValueError("Input text is empty. Please pass a valid transcript.")

    # ── Step 1: Chunk the transcript ─────────────────────────
    print(f"[1/5] Splitting transcript into overlapping chunks")
    print(f"      (window={sentences_per_chunk} sentences, overlap={overlap})…")
    chunks = _chunk_text_for_qa(text, sentences_per_chunk, overlap)

    if not chunks:
        raise ValueError("No chunks were produced. The transcript may be too short.")

    print(f"      {len(chunks)} chunks created.")

    # ── Step 2: Build DataFrame ──────────────────────────────
    print(f"[2/5] Building DataFrame and saving to '{chunks_file}'…")
    df = pd.DataFrame({
        "chunk_id": range(len(chunks)),
        "text":     chunks,
        "word_count": [len(c.split()) for c in chunks]
    })
    df.to_csv(chunks_file, index=False, encoding="utf-8")
    print(f"      Saved {len(df)} rows → {chunks_file}")
    print(f"      Average chunk length: {df['word_count'].mean():.1f} words")

    print("\n" + "=" * 60)
    print("TASK C — VECTORIZATION & FAISS INDEX")
    print("=" * 60)

    # ── Step 3: Load embedding model ─────────────────────────
    print("[3/5] Loading embedding model (all-MiniLM-L6-v2)…")
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print("      Model loaded.")

    # ── Step 4: Encode chunks → embeddings ───────────────────
    print(f"[4/5] Encoding {len(chunks)} chunks into embeddings…")
    embeddings = encoder.encode(
        chunks,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    # FAISS requires float32
    embeddings = embeddings.astype(np.float32)

    embedding_dim = embeddings.shape[1]
    print(f"      Embeddings shape: {embeddings.shape}  (dim={embedding_dim})")

    # ── Step 5: Build & save FAISS index ─────────────────────
    print(f"[5/5] Building FAISS IndexFlatL2 (dim={embedding_dim}) and saving…")
    index = faiss.IndexFlatL2(embedding_dim)
    index.add(embeddings)
    faiss.write_index(index, index_file)

    print(f"      Vectors in index : {index.ntotal}")
    print(f"      Index saved to   : {index_file}")
    print("=" * 60 + "\n")

    return df, index



In [16]:
def extract_lecture_text(transcript_path: str = "transcript.txt") -> str:
    """
    Read a pre-saved transcript from a .txt file.

    Args:
        transcript_path: Path to the transcript text file (default: transcript.txt).

    Returns:
        Clean transcript string ready for Module 2.
    """
    print(f"[1/2] Reading transcript from '{transcript_path}'...")

    try:
        with open(transcript_path, "r", encoding="utf-8") as f:
            raw_text = f.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"Transcript file not found: '{transcript_path}'")

    if not raw_text.strip():
        raise ValueError("Transcript file is empty.")

    clean = _clean_text(raw_text)

    print(f"[2/2] Done. Extracted {len(clean.split())} words.")
    return clean

In [17]:
# ════════════════════════════════════════════════════════════
# QUICK DEMO  (run this cell in Colab to verify everything works)
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # ── Paste or load your transcript here ──────────────────
    # from module1_data_extractor import extract_lecture_text
    # transcript = extract_lecture_text("https://www.youtube.com/watch?v=YOUR_ID")

    # Minimal demo text (replace with the real transcript)
    transcript = extract_lecture_text("transcript.txt")
       # repeat to simulate a longer lecture

    # ── Run Module 2 ─────────────────────────────────────────
    summary   = generate_lecture_summary(transcript)
    df, index = create_faiss_index(transcript)

    print("All outputs generated:")
    print("  • summary.txt")
    print("  • lecture_chunks.csv")
    print("  • lecture_index.index")

[1/2] Reading transcript from 'transcript.txt'...
[2/2] Done. Extracted 730 words.
TASK A - SUMMARIZATION
[1/4] Loading tokenizer and model (facebook/bart-large-cnn)...


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

      Model loaded.
[4/5] Encoding 20 chunks into embeddings…


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

      Embeddings shape: (20, 384)  (dim=384)
[5/5] Building FAISS IndexFlatL2 (dim=384) and saving…
      Vectors in index : 20
      Index saved to   : lecture_index.index

All outputs generated:
  • summary.txt
  • lecture_chunks.csv
  • lecture_index.index
